In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install torch torchvision --quiet
!pip install nibabel tqdm wandb --quiet
!pip -q install monai timm nibabel --quiet

In [ ]:
# Step 3: Import Libraries

# Core
import os
import numpy as np
import random

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Torchvision (for transforms, image utilities)
import torchvision.transforms as transforms

# Medical imaging
import nibabel as nib  # for NIfTI (.nii.gz) CT scans

# Visualization
import matplotlib.pyplot as plt
import cv2  # for handling PNG X-rays
from tqdm import tqdm  # for progress bars
from scipy.ndimage import zoom

# Experiment tracking
import wandb


In [ ]:
import os, glob
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import nibabel as nib
import numpy as np
import torch.nn.functional as F
from torchvision import transforms as T


class SmallBoneDataset(Dataset):
    def __init__(
        self,
        root_dir,
        img_size_2d=(128, 128),          # H, W for x-rays
        img_size_3d=(64, 128, 128),      # D, H, W for volume
        transform_2d=None,
        require_lat=False,                # if True, drop samples without lateral
        return_volume=True,               # toggle if you don't need the 3D yet
        duplicate_ap_if_missing_lat=True  # use AP as XB when lat is missing
    ):
        """
        root_dir:/content/drive/MyDrive/total_segmentator_refine/femur
        """
        self.root_dir = root_dir
        self.img_size_2d = img_size_2d
        self.img_size_3d = img_size_3d
        self.return_volume = return_volume
        self.require_lat = require_lat
        self.duplicate_ap_if_missing_lat = duplicate_ap_if_missing_lat

        # default 2D transform: grayscale -> [0,1] tensor -> resize (H,W)
        self.transform_2d = transform_2d or T.Compose([
            T.Grayscale(num_output_channels=1),
            T.ToTensor(),                       # [1, H, W] in [0,1]
            T.Resize(img_size_2d, antialias=True),
        ])

        # Build an index of valid samples upfront (robust to missing files)
        self.samples = []
        for pdir in sorted(glob.glob(os.path.join(root_dir, "p_*"))):  # changed from CLINIC_* to p_*
            ap = glob.glob(os.path.join(pdir, "x-ray", "*ap*.png"))
            lat = glob.glob(os.path.join(pdir, "x-ray", "*lat*.png"))
            vol = glob.glob(os.path.join(pdir, "ct-scan", "*.nii.gz"))

            if not ap or not vol:
                continue
            if self.require_lat and not lat:
                continue

            self.samples.append({
                "pid": os.path.basename(pdir),
                "ap": ap[0],
                "lat": lat[0] if lat else None,
                "vol": vol[0],
            })

        if len(self.samples) == 0:
            raise RuntimeError(f"No valid samples found under {root_dir}")

    def __len__(self):
        return len(self.samples)

    def _safe_minmax(self, a, eps=1e-8):
        a = a.astype(np.float32)
        amin, amax = float(np.min(a)), float(np.max(a))
        if not np.isfinite(amin) or not np.isfinite(amax) or (amax - amin) < eps:
            return np.zeros_like(a, dtype=np.float32)
        return (a - amin) / (amax - amin + eps)

    def _load_volume_resized(self, path):
        vol_np = nib.load(path).get_fdata(dtype=np.float32)
        vol_np = self._safe_minmax(vol_np)                     # [D0, H0, W0] in [0,1]

        vol = torch.from_numpy(vol_np).unsqueeze(0).unsqueeze(0)  # [1,1,D0,H0,W0]
        D, H, W = self.img_size_3d
        vol = F.interpolate(vol, size=(D, H, W), mode="trilinear", align_corners=False)
        return vol.squeeze(0)  # [1, D, H, W]

    def _load_xray(self, path):
        img = Image.open(path).convert("L")
        return self.transform_2d(img)  # [1, H, W] in [0,1]

    def __getitem__(self, idx):
        item = self.samples[idx]
        pid = item["pid"]

        # ---- 2D inputs (xa=AP, xb=LAT or fallback) ----
        xa = self._load_xray(item["ap"])
        if item["lat"] is not None:
            xb = self._load_xray(item["lat"])
        else:
            if self.duplicate_ap_if_missing_lat:
                xb = xa.clone()
            else:
                xb = torch.zeros_like(xa)

        # ---- 3D volume (optional) ----
        vol = self._load_volume_resized(item["vol"]) if self.return_volume else None

        # Return a handy dict; DataLoader default collate works
        return {
            "pid": pid,
            "xa": xa,           # [1, 128, 128]
            "xb": xb,           # [1, 128, 128]
            "vol": vol,         # [1, 64, 128, 128] or None
            "paths": item,
        }


In [ ]:
import torch
from torch.utils.data import random_split, DataLoader

# -------- Collate function (stacks tensors, keeps metadata as lists) --------
def collate_small(batch):
    xa  = torch.stack([b["xa"]  for b in batch], dim=0)                # [B,1,128,128]
    xb  = torch.stack([b["xb"]  for b in batch], dim=0)                # [B,1,128,128]
    vol = None
    if batch[0]["vol"] is not None:
        vol = torch.stack([b["vol"] for b in batch], dim=0)            # [B,1,64,128,128]
    pids  = [b["pid"]   for b in batch]
    paths = [b["paths"] for b in batch]
    return {"pid": pids, "xa": xa, "xb": xb, "vol": vol, "paths": paths}

# -------- Dataset size & splits --------
small_total_size = len(small_dataset)
print("Total patients:", small_total_size)

small_train_size = int(0.7 * small_total_size)
small_val_size   = int(0.2 * small_total_size)
small_test_size  = small_total_size - small_train_size - small_val_size

print(f"Train: {small_train_size}, Val: {small_val_size}, Test: {small_test_size}")

small_generator = torch.Generator().manual_seed(42)
small_train_set, small_val_set, small_test_set = random_split(
    small_dataset,
    [small_train_size, small_val_size, small_test_size],
    generator=small_generator
)

# -------- Choose batch size (1 for 3D volumes to be safe) --------
batch_size = 1 if getattr(small_dataset, "return_volume", True) else 8

# -------- DataLoaders --------
small_train_loader = DataLoader(
    small_train_set, batch_size=batch_size, shuffle=True,
    num_workers=2, pin_memory=True, collate_fn=collate_small
)
small_val_loader = DataLoader(
    small_val_set, batch_size=batch_size, shuffle=False,
    num_workers=2, pin_memory=True, collate_fn=collate_small
)
small_test_loader = DataLoader(
    small_test_set, batch_size=batch_size, shuffle=False,
    num_workers=2, pin_memory=True, collate_fn=collate_small
)

# -------- Inspect one batch --------
for batch in small_train_loader:
    xa, xb, vol = batch["xa"], batch["xb"], batch["vol"]
    print("AP batch  :", tuple(xa.shape))                # e.g. (B,1,128,128)
    print("LAT batch :", tuple(xb.shape))                # e.g. (B,1,128,128)
    print("CT batch  :", tuple(vol.shape) if vol is not None else None)  # e.g. (B,1,64,128,128)
    print("PIDs      :", batch["pid"][:3])
    break


In [ ]:
# Import
import wandb

# Login with your API key (keep this private!)
!wandb login 47db746f38dd6216f60f4b6ce9c5d98e507dd6a3

In [ ]:
from typing import MutableMapping, Sequence, Tuple, Union
import math
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

from timm.models.layers import DropPath, to_2tuple, trunc_normal_
from monai.networks.blocks.dynunet_block import UnetBasicBlock, UnetResBlock, get_conv_layer

# ------------------------------------------------------------
# UNETR-style 3D building blocks
# ------------------------------------------------------------

FeaturesDictType = MutableMapping[str, torch.Tensor]

class UnetrUpBlock(nn.Module):
    def __init__(self, spatial_dims: int, in_channels: int, out_channels: int,
                 kernel_size: int, upsample_kernel_size: int, norm_name: str, res_block: bool = False):
        super().__init__()
        self.transp_conv = get_conv_layer(
            spatial_dims, in_channels, out_channels,
            kernel_size=upsample_kernel_size, stride=upsample_kernel_size,
            conv_only=True, is_transposed=True
        )
        if res_block:
            self.conv_block = UnetResBlock(spatial_dims, out_channels + out_channels, out_channels,
                                           kernel_size=kernel_size, stride=1, norm_name=norm_name)
        else:
            self.conv_block = UnetBasicBlock(spatial_dims, out_channels + out_channels, out_channels,
                                             kernel_size=kernel_size, stride=1, norm_name=norm_name)
    def forward(self, x, skip):
        x = self.transp_conv(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv_block(x)
        return x

class UnetrBasicBlock(nn.Module):
    def __init__(self, spatial_dims: int, in_channels: int, out_channels: int,
                 kernel_size: int, stride: int, norm_name: str, res_block: bool = False):
        super().__init__()
        if res_block:
            self.layer = UnetResBlock(spatial_dims, in_channels, out_channels,
                                      kernel_size=kernel_size, stride=stride, norm_name=norm_name)
        else:
            self.layer = UnetBasicBlock(spatial_dims, in_channels, out_channels,
                                        kernel_size=kernel_size, stride=stride, norm_name=norm_name)
    def forward(self, x):
        return self.layer(x)

class UnetOutBlock(nn.Module):
    def __init__(self, spatial_dims: int, in_channels: int, out_channels: int, dropout: float = None):
        super().__init__()
        self.conv = get_conv_layer(spatial_dims, in_channels, out_channels, kernel_size=1, stride=1,
                                   dropout=dropout, bias=True, act=None, norm=None, conv_only=False)
    def forward(self, x):
        return self.conv(x)

# ------------------------------------------------------------
# Swin Transformer (2D) pieces
# ------------------------------------------------------------

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        x = self.fc1(x); x = self.act(x); x = self.drop(x); x = self.fc2(x); x = self.drop(x); return x

def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size[0], window_size[0], W // window_size[1], window_size[1], C)
    windows = x.permute(0,1,3,2,4,5).contiguous().view(-1, window_size[0], window_size[1], C)
    return windows

def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / window_size[0] / window_size[1]))
    x = windows.view(B, H // window_size[0], W // window_size[1], window_size[0], window_size[1], -1)
    x = x.permute(0,1,3,2,4,5).contiguous().view(B, H, W, -1)
    return x

class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, qk_scale=None, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5

        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2*window_size[0]-1)*(2*window_size[1]-1), num_heads)
        )

        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        # ✅ avoid warning
        coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing='ij'))  # 2, Wh, Ww
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += self.window_size[0] - 1
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer("relative_position_index", relative_position_index)

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        trunc_normal_(self.relative_position_bias_table, std=.02)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))

        rel_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0]*self.window_size[1], self.window_size[0]*self.window_size[1], -1
        ).permute(2, 0, 1).contiguous()
        attn = attn + rel_bias.unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x
class SwinTransformerBlock(nn.Module):
    def __init__(self, dim, input_resolution, num_heads, window_size, shift_size,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0., drop_path=0.,
                 act_layer=nn.GELU, norm_layer=nn.LayerNorm, fused_window_process=False):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = list(window_size)
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio

        if min(self.input_resolution) <= min(self.window_size):
            self.shift_size = [0, 0]
            self.window_size[0] = self.input_resolution[0]
            self.window_size[1] = self.input_resolution[1]
        assert 0 <= self.shift_size[0] < self.window_size[0]
        assert 0 <= self.shift_size[1] < self.window_size[1]

        self.norm1 = norm_layer(dim)
        # ⚠️ make sure the rest of your code imports this patched WindowAttention
        self.attn = WindowAttention(dim, window_size=to_2tuple(self.window_size), num_heads=num_heads,
                                    qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)
        self.drop_path = nn.Identity() if drop_path == 0. else DropPath(drop_path)
        self.norm2 = norm_layer(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            act_layer(),
            nn.Dropout(drop),
            nn.Linear(int(dim * mlp_ratio), dim),
            nn.Dropout(drop),
        )

        # ✅ FIX: build mask with true ceil-to-window, not "+1" unconditionally
        if self.shift_size[0] > 0 or self.shift_size[1] > 0:
            H, W = self.input_resolution
            Hp = int(math.ceil(H / self.window_size[0])) * self.window_size[0]
            Wp = int(math.ceil(W / self.window_size[1])) * self.window_size[1]

            img_mask = torch.zeros((1, Hp, Wp, 1))
            h_slices = (slice(0, -self.window_size[0]),
                        slice(-self.window_size[0], -self.shift_size[0]),
                        slice(-self.shift_size[0], None))
            w_slices = (slice(0, -self.window_size[1]),
                        slice(-self.window_size[1], -self.shift_size[1]),
                        slice(-self.shift_size[1], None))
            cnt = 0
            for h in h_slices:
                for w in w_slices:
                    img_mask[:, h, w, :] = cnt
                    cnt += 1

            from torch.nn.functional import pad as fpad
            # If Hp/H or Wp/W added padding, that's fine—the forward also pads consistently.
            mask_windows = img_mask.unfold(1, self.window_size[0], self.window_size[0]) \
                                     .unfold(2, self.window_size[1], self.window_size[1]) \
                                     .contiguous().view(-1, self.window_size[0] * self.window_size[1])
            # The above unfold approach is equivalent to window_partition for zeros.
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, 0.0)
        else:
            attn_mask = None

        self.register_buffer("attn_mask", attn_mask)
        self.fused_window_process = fused_window_process

    def forward(self, x):
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W

        shortcut = x
        x = self.norm1(x).view(B, H, W, C)

        pad_b = (self.window_size[0] - H % self.window_size[0]) % self.window_size[0]
        pad_r = (self.window_size[1] - W % self.window_size[1]) % self.window_size[1]
        x = F.pad(x, (0, 0, 0, pad_r, 0, pad_b))
        _, Hp, Wp, _ = x.shape

        if self.shift_size[0] > 0 or self.shift_size[1] > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size[0], -self.shift_size[1]), dims=(1, 2))
        else:
            shifted_x = x

        # partition windows
        x_windows = shifted_x.unfold(1, self.window_size[0], self.window_size[0]) \
                             .unfold(2, self.window_size[1], self.window_size[1]) \
                             .contiguous().view(-1, self.window_size[0]*self.window_size[1], C)

        # W-MSA/SW-MSA
        attn_windows = self.attn(x_windows, mask=self.attn_mask) \
                         .view(-1, self.window_size[0], self.window_size[1], C)

        # merge windows
        B_ = B * (Hp // self.window_size[0]) * (Wp // self.window_size[1])
        x = attn_windows.view(B_, self.window_size[0], self.window_size[1], C) \
                         .contiguous().view(B, Hp, Wp, C)

        if self.shift_size[0] > 0 or self.shift_size[1] > 0:
            x = torch.roll(x, shifts=(self.shift_size[0], self.shift_size[1]), dims=(1, 2))

        if pad_r > 0 or pad_b > 0:
            x = x[:, :H, :W, :].contiguous()

        x = x.view(B, H * W, C)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x

class PatchMerging(nn.Module):
    def __init__(self, input_resolution, dim, norm_layer=nn.LayerNorm):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(4 * dim)
    def forward(self, x):
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W and H % 2 == 0 and W % 2 == 0
        x = x.view(B, H, W, C)
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x = torch.cat([x0, x1, x2, x3], dim=-1).view(B, -1, 4 * C)
        x = self.norm(x)
        x = self.reduction(x)
        return x

class BasicLayer(nn.Module):
    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.LayerNorm, downsample=None, fused_window_process=False):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                dim=dim, input_resolution=input_resolution, num_heads=num_heads,
                window_size=window_size,
                shift_size=[0, 0] if (i % 2 == 0) else [window_size[0] // 2, window_size[1] // 2],
                mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop, attn_drop=attn_drop, drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                norm_layer=norm_layer, fused_window_process=fused_window_process
            ) for i in range(depth)
        ])
        self.norm = nn.LayerNorm(dim * 2)
        self.downsample = downsample(input_resolution, dim=dim, norm_layer=norm_layer) if downsample is not None else None

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        if self.downsample is not None:
            x = self.downsample(x)
        x_out = self.norm(x).reshape(-1, self.input_resolution[0] // 2, self.input_resolution[1] // 2, self.dim * 2)
        return x, x_out

class PatchEmbed(nn.Module):
    def __init__(self, img_size=(160, 224), patch_size=2, in_chans=1, embed_dim=96, norm_layer=nn.LayerNorm):
        super().__init__()
        img_size = to_2tuple(img_size); patch_size = to_2tuple(patch_size)
        patches_resolution = [img_size[0] // patch_size[0], img_size[1] // patch_size[1]]
        self.img_size = img_size
        self.patch_size = patch_size
        self.patches_resolution = patches_resolution
        self.in_chans = in_chans
        self.embed_dim = embed_dim
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = norm_layer(embed_dim)

    def forward(self, x):
        B, C, H, W = x.shape
        assert (H, W) == (self.img_size[0], self.img_size[1]), \
            f"Input size ({H}x{W}) != model ({self.img_size[0]}x{self.img_size[1]})."
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x

class SwinTransformer(nn.Module):
    def __init__(self, img_size=(160, 224), patch_size=2, in_chans=1, embed_dim=96,
                 depths=(2, 2, 18, 2), num_heads=(3, 6, 12, 24),
                 window_size=(8, 8), mlp_ratio=4., qkv_bias=True, qk_scale=None,
                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0.2,
                 norm_layer=nn.LayerNorm, ape=False, patch_norm=True, fused_window_process=False, **kwargs):
        super().__init__()
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.ape = ape
        self.patch_norm = patch_norm
        self.mlp_ratio = mlp_ratio
        self.img_size = img_size
        self.patch_size = patch_size

        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size, in_chans=in_chans,
                                      embed_dim=embed_dim, norm_layer=norm_layer if self.patch_norm else None)
        num_patches = self.patch_embed.patches_resolution[0] * self.patch_embed.patches_resolution[1]
        patches_resolution = self.patch_embed.patches_resolution
        self.patches_resolution = patches_resolution

        if self.ape:
            self.absolute_pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
            trunc_normal_(self.absolute_pos_embed, std=.02)

        self.pos_drop = nn.Dropout(p=drop_rate)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]

        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            layer = BasicLayer(
                dim=int(embed_dim * 2 ** i_layer),
                input_resolution=(patches_resolution[0] // (2 ** i_layer),
                                  patches_resolution[1] // (2 ** i_layer)),
                depth=depths[i_layer],
                num_heads=num_heads[i_layer],
                window_size=window_size,
                mlp_ratio=self.mlp_ratio,
                qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop_rate, attn_drop=attn_drop_rate,
                drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],
                norm_layer=norm_layer,
                downsample=PatchMerging,
                fused_window_process=fused_window_process
            )
            self.layers.append(layer)

        self.norm = nn.LayerNorm(self.embed_dim)

    def forward(self, x):
        x = self.patch_embed(x)
        if self.ape:
            x = x + self.absolute_pos_embed
        x = self.pos_drop(x)

        out_list = []
        x_mid = self.norm(x).reshape(
            -1,
            self.img_size[0] // self.patch_size,
            self.img_size[1] // self.patch_size,
            self.embed_dim
        )
        out_list.append(x_mid)  # stage -1 (embed level)

        for i in range(self.num_layers):
            x, x_mid = self.layers[i](x)
            out_list.append(x_mid)  # stages 0..3

        return out_list

# ------------------------------------------------------------
# Cross-view TransFusion (3D) pieces
# ------------------------------------------------------------

class Attention(nn.Module):
    def __init__(self, num_heads=8, hidden_size=768, atte_dropout_rate=0.0):
        super().__init__()
        self.num_attention_heads = num_heads
        self.attention_head_size = int(hidden_size / self.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size
        self.query = nn.Linear(hidden_size, self.all_head_size)
        self.key   = nn.Linear(hidden_size, self.all_head_size)
        self.value = nn.Linear(hidden_size, self.all_head_size)
        self.out = nn.Linear(hidden_size, hidden_size)
        self.attn_dropout = nn.Dropout(atte_dropout_rate)
        self.proj_dropout = nn.Dropout(atte_dropout_rate)
        self.softmax = nn.Softmax(dim=-1)

    def _reshape(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, x1, x2):
        q1, k1, v1 = self._reshape(self.query(x1)), self._reshape(self.key(x1)), self._reshape(self.value(x1))
        q2, k2, v2 = self._reshape(self.query(x2)), self._reshape(self.key(x2)), self._reshape(self.value(x2))

        attn1 = self.softmax((q1 @ k2.transpose(-1, -2)) / math.sqrt(self.attention_head_size))
        attn1 = self.attn_dropout(attn1)
        ctx1 = (attn1 @ v2).permute(0, 2, 1, 3).contiguous().view(x1.size(0), x1.size(1), -1)
        out1 = self.proj_dropout(self.out(ctx1))

        attn2 = self.softmax((q2 @ k1.transpose(-1, -2)) / math.sqrt(self.attention_head_size))
        attn2 = self.attn_dropout(attn2)
        ctx2 = (attn2 @ v1).permute(0, 2, 1, 3).contiguous().view(x2.size(0), x2.size(1), -1)
        out2 = self.proj_dropout(self.out(ctx2))

        return out1, out2

class Block(nn.Module):
    def __init__(self, hidden_size=768, mlp_dim=1536, dropout_rate=0.5, num_heads=8, atte_dropout_rate=0.0):
        super().__init__()
        self.attention_norm = nn.LayerNorm(hidden_size, eps=1e-6)
        self.attn = Attention(num_heads=num_heads, hidden_size=hidden_size, atte_dropout_rate=atte_dropout_rate)
    def forward(self, x1, x2):
        x1 = self.attention_norm(x1)
        x2 = self.attention_norm(x2)
        x1, x2 = self.attn(x1, x2)
        return x1, x2

class TransFusion(nn.Module):
    def __init__(self, hidden_size=768, num_layers=2, mlp_dim=1536, dropout_rate=0.5,
                 num_heads=8, atte_dropout_rate=0.0, roi_size: Union[Sequence[int], int] = (64,64,64), scale=16):
        super().__init__()
        if isinstance(roi_size, int):
            roi_size = [roi_size, roi_size, roi_size]
        patch_size = (1,1,1)
        self.size = (roi_size[0] // patch_size[0] // scale,
                     roi_size[1] // patch_size[1] // scale,
                     roi_size[2] // patch_size[2] // scale)
        n_patches = self.size[0] * self.size[1] * self.size[2]
        self.encoder_norm = nn.LayerNorm(hidden_size, eps=1e-6)
        self.patch_embeddings = nn.Conv3d(hidden_size, hidden_size, kernel_size=patch_size, stride=patch_size)
        self.position_embeddings = nn.Parameter(torch.zeros(1, n_patches, hidden_size))
        self.dropout = nn.Dropout(dropout_rate)
        self.layer = nn.ModuleList([
            Block(hidden_size=hidden_size, mlp_dim=mlp_dim, dropout_rate=dropout_rate,
                  num_heads=num_heads, atte_dropout_rate=atte_dropout_rate)
            for _ in range(num_layers)
        ])

    def forward(self, x1, x2):
        x1 = self.patch_embeddings(x1)
        x2 = self.patch_embeddings(x2)
        x1 = x1.flatten(2).transpose(-1, -2)
        x2 = x2.flatten(2).transpose(-1, -2)
        x1 = self.dropout(x1 + self.position_embeddings)
        x2 = self.dropout(x2 + self.position_embeddings)
        for blk in self.layer:
            x1, x2 = blk(x1, x2)
        x1 = self.encoder_norm(x1)
        x2 = self.encoder_norm(x2)
        B, N, hidden = x1.size()
        x1 = x1.permute(0, 2, 1).contiguous().view(B, hidden, self.size[0], self.size[1], self.size[2])
        x2 = x2.permute(0, 2, 1).contiguous().view(B, hidden, self.size[0], self.size[1], self.size[2])
        return x1, x2

# ------------------------------------------------------------
# SwinX2S (two 2D views -> single 3D fused output)
# ------------------------------------------------------------


class SwinX2S(nn.Module):
    """
    Inputs:
      xa, xb: [B, 1, H, W]  (H,W == img_size[1:])
    Output:
      fused:  [B, out_channels, D, H, W]
    """
    def __init__(self,
                 img_size: Union[Sequence[int], int],
                 in_channels: int,
                 out_channels: int,
                 feature_size: int = 96,
                 depths: Sequence[int] = (2, 2, 18, 2),
                 num_heads: Sequence[int] = (3, 6, 12, 24),
                 norm_name: Union[Tuple, str] = "instance",
                 drop_rate: float = 0.0,
                 attn_drop_rate: float = 0.0,
                 spatial_dims: int = 3,
                 fusion_depths: Sequence[int] = (2, 2, 2, 2, 2, 2),
                 window_size: Sequence[int] = (8, 8)) -> None:
        super().__init__()
        assert isinstance(img_size, (list, tuple)) and len(img_size) == 3, "img_size must be [D,H,W]"
        D, H, W = img_size
        assert D % 32 == 0, "Depth D should be divisible by 32."
        self.img_size = img_size

        # 3D encoders (✅ fix: encoder1 takes in_channels)
        self.encoder1 = UnetrBasicBlock(spatial_dims, in_channels,        feature_size, 3, 1, norm_name, res_block=True)
        self.encoder2 = UnetrBasicBlock(spatial_dims, feature_size,       feature_size, 3, 1, norm_name, res_block=True)
        self.encoder3 = UnetrBasicBlock(spatial_dims, 2*feature_size,     2*feature_size, 3, 1, norm_name, res_block=True)
        self.encoder4 = UnetrBasicBlock(spatial_dims, 4*feature_size,     4*feature_size, 3, 1, norm_name, res_block=True)
        self.encoder5 = UnetrBasicBlock(spatial_dims, 8*feature_size,     8*feature_size, 3, 1, norm_name, res_block=True)
        self.encoder10= UnetrBasicBlock(spatial_dims, 16*feature_size,    16*feature_size,3, 1, norm_name, res_block=True)

        # 3D decoders
        self.decoder5 = UnetrUpBlock(spatial_dims, 16*feature_size, 8*feature_size, 3, 2, norm_name, res_block=True)
        self.decoder4 = UnetrUpBlock(spatial_dims,  8*feature_size, 4*feature_size, 3, 2, norm_name, res_block=True)
        self.decoder3 = UnetrUpBlock(spatial_dims,  4*feature_size, 2*feature_size, 3, 2, norm_name, res_block=True)
        self.decoder2 = UnetrUpBlock(spatial_dims,  2*feature_size,   feature_size, 3, 2, norm_name, res_block=True)
        self.decoder1 = UnetrUpBlock(spatial_dims,    feature_size,   feature_size, 3, 2, norm_name, res_block=True)
        self.out = UnetOutBlock(spatial_dims, feature_size, out_channels)

        # deep cross-attention
        self.cross_atte6 = TransFusion(hidden_size=feature_size*16, num_layers=fusion_depths[5],
                                       mlp_dim=feature_size*32, num_heads=num_heads[3],
                                       dropout_rate=drop_rate, atte_dropout_rate=attn_drop_rate,
                                       roi_size=img_size, scale=32)

        # shared 2D Swin
        self.swinViT = SwinTransformer(in_chans=in_channels, embed_dim=feature_size, depths=depths,
                                       img_size=img_size[1:], window_size=window_size,
                                       patch_size=2, num_heads=num_heads)

        # 2D -> 3D inflation (Conv3d(1 -> D_i))
        self.expand_ = nn.ModuleList([nn.Conv3d(1, int(img_size[0] / (2 ** i)), 1)
                                      for i in range(len(depths) + 2)])
        for e in self.expand_:
            nn.init.constant_(e.weight, 1.0)
            nn.init.constant_(e.bias, 0.0)

        # final fusion head
        self.fuse = nn.Conv3d(out_channels * 2, out_channels, kernel_size=1, bias=True)

    def _inflate_list_to_3d(self, feats_2d_list, *, pos: bool):
        out = []
        for i, x in enumerate(feats_2d_list):
            x = x.permute(0, 3, 1, 2)      # [B, c, h, w]
            x = x.unsqueeze(1)             # [B, 1, c, h, w]
            x = self.expand_[i](x)         # ✅ remove .half() here
            x = x.permute(0, 2, 1, 3, 4)   # [B, c, D_i, h, w]
            out.append(x)
        return out


    def forward_view_encoder(self, x, pos=True):
        # Swin features: list of [B,h,w,c]
        feats = self.swinViT(x)
        feats.insert(0, x.permute(0, 2, 3, 1))   # prepend raw input as [B,H,W,1]
        vols = self._inflate_list_to_3d(feats, pos=pos)  # list of [B,c,D_i,H,W]

        x_enc0 = self.encoder1(vols[0])  # in_channels = 1  -> out 96
        x_enc1 = self.encoder2(vols[1])  # in_channels = 96 -> out 96
        x_enc2 = self.encoder3(vols[2])  # 192 -> 192
        x_enc3 = self.encoder4(vols[3])  # 384 -> 384
        x_enc4 = self.encoder5(vols[4])  # 768 -> 768
        x_dec4 = self.encoder10(vols[5]) # 1536-> 1536
        return {"enc0": x_enc0, "enc1": x_enc1, "enc2": x_enc2, "enc3": x_enc3, "enc4": x_enc4, "dec4": x_dec4}

    def forward_view_decoder(self, feats: FeaturesDictType):
        x = self.decoder5(feats["dec4"], feats["enc4"])
        x = self.decoder4(x, feats["enc3"])
        x = self.decoder3(x, feats["enc2"])
        x = self.decoder2(x, feats["enc1"])
        x = self.decoder1(x, feats["enc0"])
        return self.out(x)

    def forward(self, xa: torch.Tensor, xb: torch.Tensor):
        xa_e = self.forward_view_encoder(xa, pos=True)
        xb_e = self.forward_view_encoder(xb, pos=False)
        xa_e["dec4"], xb_e["dec4"] = self.cross_atte6(xa_e["dec4"], xb_e["dec4"])
        ya = self.forward_view_decoder(xa_e)  # [B,C,D,H,W]
        yb = self.forward_view_decoder(xb_e)  # [B,C,D,H,W]
        y = torch.cat([ya, yb], dim=1)
        return self.fuse(y)

In [ ]:
# Tiny, Colab-friendly config
IMG_SIZE = [64, 128, 128]      # D,H,W  (D divisible by 32; 64 is OK)
FEATURE  = 48                  # feature channels
DEPTHS   = (2, 2, 2, 2)        # fewer Swin stages
HEADS    = (2, 4, 6, 8)        # attention heads
WINDOW   = (8, 8)              # Swin window size

# Create model in full precision (float32)
model = SwinX2S(
    img_size=IMG_SIZE,
    in_channels=1,
    out_channels=1,
    feature_size=FEATURE,
    depths=DEPTHS,
    num_heads=HEADS,
    window_size=WINDOW,
    norm_name="batch",
).to("cuda")     # ✅ keep full precision, just move to GPU

# Make small random inputs (float32)
B, H, W = 1, IMG_SIZE[1], IMG_SIZE[2]
xa = torch.randn(B, 1, H, W, device="cuda", dtype=torch.float32)
xb = torch.randn(B, 1, H, W, device="cuda", dtype=torch.float32)

# Run inference (no gradients, mixed precision safe)
model.eval()
with torch.no_grad(), torch.amp.autocast('cuda'):
    y = model(xa, xb)

print("Output shape:", tuple(y.shape))


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =========================
# Loss Function (BCE + Dice)
# =========================
class DiceBCELoss(nn.Module):
    """
    Inputs:
      preds: logits [B, C, H, W] or [B, C, D, H, W]  (C usually = 1)
      targets: same shape as preds (float {0,1} or probabilities in [0,1])

    Args:
      bce_weight: weight for BCE term
      dice_weight: weight for Dice term
      from_logits: if True, BCE uses logits; otherwise expects probabilities
      pos_weight: optional tensor for positive class weighting in BCE (handles imbalance)
      smooth: smoothing for Dice
      eps: numerical stability
    """
    def __init__(self,
                 bce_weight: float = 0.5,
                 dice_weight: float = 0.5,
                 from_logits: bool = True,
                 pos_weight: torch.Tensor | None = None,
                 smooth: float = 1.0,
                 eps: float = 1e-6):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.from_logits = from_logits
        self.pos_weight = pos_weight
        self.smooth = smooth
        self.eps = eps

    def forward(self, preds: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # Ensure shapes match
        if preds.shape != targets.shape:
            raise ValueError(f"DiceBCELoss: preds shape {tuple(preds.shape)} != targets shape {tuple(targets.shape)}")

        # Targets to float
        targets = targets.float()

        # ---- BCE term (compute in fp32 for stability) ----
        if self.from_logits:
            bce = F.binary_cross_entropy_with_logits(
                preds.float(), targets, pos_weight=self.pos_weight, reduction="mean"
            )
            probs = torch.sigmoid(preds.float())
        else:
            bce = F.binary_cross_entropy(
                preds.float().clamp_(0, 1), targets, reduction="mean"
            )
            probs = preds.float().clamp_(0, 1)

        # ---- Dice term (soft dice over all spatial dims; mean over B,C) ----
        # Flatten over spatial dims only
        dims = list(range(2, preds.ndim))  # [H,W] or [D,H,W]
        intersection = torch.sum(probs * targets, dim=dims)
        denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)
        dice_per = (2.0 * intersection + self.smooth) / (denom + self.smooth + self.eps)
        dice_loss = 1.0 - dice_per.mean()

        return self.bce_weight * bce + self.dice_weight * dice_loss


# =========================
# Metric Function (Dice)
# =========================
@torch.no_grad()
def compute_dice(preds: torch.Tensor,
                 targets: torch.Tensor,
                 from_logits: bool = True,
                 threshold: float = 0.5,
                 smooth: float = 1.0,
                 eps: float = 1e-6) -> float:
    """
    Returns mean Dice over batch (and channels if present).
    Works for 2D or 3D. If from_logits=True, applies sigmoid first.
    """
    targets = targets.float()
    if from_logits:
        preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()

    dims = list(range(2, preds.ndim))
    intersection = torch.sum(preds * targets, dim=dims)
    denom = torch.sum(preds, dim=dims) + torch.sum(targets, dim=dims)
    dice_per = (2.0 * intersection + smooth) / (denom + smooth + eps)
    return dice_per.mean().item()


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
import wandb

# ========= Config =========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_wandb = True
num_epochs = 20
lr = 1e-4
grad_clip = 1.0
patience = 10
save_path = "/content/drive/MyDrive/saved_models_swinX2S/best_model_01.pth"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

TARGET_KEY = "vol"   # <-- change to "mask" or other key if your target is different
BINARY_THRESH = 0.5  # for Dice metric thresholding

# ========= Loss / Optim / Scheduler =========
# Expect criterion to take logits and binary targets (BCE+Dice often expects logits)
criterion = DiceBCELoss().to(device)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

# Mixed precision
scaler = torch.cuda.amp.GradScaler(enabled=True)

# ========= Helpers =========
@torch.no_grad()
def match_target_to_output(target, out):
    """
    Align target to out shape.
    - out: [B,1,H,W] or [B,1,D,H,W]
    - target: [B,1,H,W] or [B,1,D,H,W]
    Returns: float tensor in [0,1] with same shape as out.
    """
    target = target.float()

    # Case A: model output is 3D
    if out.ndim == 5:
        B, C, D, H, W = out.shape
        if target.ndim == 4:  # [B,1,H,W] -> add depth=1 then interpolate
            target = target.unsqueeze(2)  # [B,1,1,H,W]
        elif target.ndim != 5:
            raise ValueError(f"Unexpected target ndim={target.ndim} for 3D output")

        if target.shape[-3:] != (D, H, W):
            target = F.interpolate(target, size=(D, H, W), mode="trilinear", align_corners=False)
        return target.clamp_(0, 1)

    # Case B: model output is 2D
    elif out.ndim == 4:
        B, C, H, W = out.shape
        if target.ndim == 5:                   # project 3D → 2D when model is 2D
            target = target.max(dim=2, keepdim=False).values  # [B,1,H,W]
        elif target.ndim != 4:
            raise ValueError(f"Unexpected target ndim={target.ndim} for 2D output")

        if target.shape[-2:] != (H, W):
            target = F.interpolate(target, size=(H, W), mode="bilinear", align_corners=False)
        return target.clamp_(0, 1)

    else:
        raise ValueError(f"Unexpected output ndim={out.ndim}")


def dice_from_logits(logits, target, thr=BINARY_THRESH, eps=1e-6):
    """
    logits -> probs -> binarize -> dice with target (assumes binary target in {0,1}).
    """
    probs = torch.sigmoid(logits)
    pred = (probs > thr).float()
    target = (target > 0.5).float()
    inter = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2 * inter + eps) / (union + eps)
    return dice.mean().item()

# ========= Tracking =========
train_epoch_losses, val_epoch_losses, val_dice_scores = [], [], []
best_val_loss = float("inf")
early_stop_counter = 0

if use_wandb:
    wandb.init(project="TransvertFinal", config={
        "epochs": num_epochs, "lr": lr, "loss": "BCE+Dice"
    })

# ========= Train/Val Loop =========
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    pbar = tqdm(small_train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for batch in pbar:
        xa = batch["xa"].to(device, non_blocking=True).half()
        xb = batch["xb"].to(device, non_blocking=True).half()

        # ---- Get target ----
        if TARGET_KEY not in batch or batch[TARGET_KEY] is None:
            raise RuntimeError(
                f"Batch has no '{TARGET_KEY}' to supervise. "
                f"Set TARGET_KEY correctly or change dataset to return it."
            )
        target_raw = batch[TARGET_KEY].to(device, non_blocking=True)   # keep fp32
        # Forward (mixed precision)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            logits = model(xa, xb)                        # expected [B,1,H,W]
            target = match_target_to_output(target_raw, logits)
            loss = criterion(logits, target)

        # Backward + step
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_postfix({"train_loss": f"{loss.item():.4f}"})

        if use_wandb:
            wandb.log({"train/batch_loss": loss.item()})

    avg_train_loss = running_loss / len(small_train_loader)
    train_epoch_losses.append(avg_train_loss)

      # ===== Validation =====
    model.eval()
    val_loss, val_dice = 0.0, 0.0
    with torch.no_grad():
        pbar_val = tqdm(small_val_loader, desc=f"Validation {epoch+1}/{num_epochs}")
        for batch in pbar_val:
            xa = batch["xa"].to(device, non_blocking=True).half()
            xb = batch["xb"].to(device, non_blocking=True).half()
            target_raw = batch[TARGET_KEY].to(device, non_blocking=True)

            with torch.cuda.amp.autocast():
                logits = model(xa, xb)
                target = match_target_to_output(target_raw, logits)
                loss = criterion(logits, target)

            val_loss += loss.item()
            val_dice += dice_from_logits(logits.float(), target.float())

            pbar_val.set_postfix({
                "val_loss": f"{loss.item():.4f}"
            })

            if use_wandb:
                wandb.log({"val/batch_loss": loss.item()})


    avg_val_loss = val_loss / len(small_val_loader)
    avg_val_dice = val_dice / len(small_val_loader)
    val_epoch_losses.append(avg_val_loss)
    val_dice_scores.append(avg_val_dice)

    # LR scheduler on validation loss
    scheduler.step(avg_val_loss)

    # Save best
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stop_counter = 0
        torch.save(model.state_dict(), save_path)
        print(f"✅ Saved new best model at {save_path}")
    else:
        early_stop_counter += 1

    # Logging
    cur_lr = optimizer.param_groups[0]["lr"]
    if use_wandb:
        wandb.log({
            "epoch": epoch + 1,
            "train/epoch_loss": avg_train_loss,
            "val/epoch_loss": avg_val_loss,
            "val/epoch_dice": avg_val_dice,
            "learning_rate": cur_lr
        })

    print(f"[Epoch {epoch+1}] "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Dice: {avg_val_dice:.4f} | "
          f"LR: {cur_lr:.2e}")

    # Early stopping
    if early_stop_counter >= patience:
        print(f"⏹️ Early stopping at epoch {epoch+1}")
        break

# ========= Plots =========
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(train_epoch_losses, label='Train Loss')
plt.plot(val_epoch_losses, label='Val Loss')
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Epoch-wise Loss"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(val_dice_scores, label='Val Dice')
plt.xlabel("Epoch"); plt.ylabel("Dice Score"); plt.title("Validation Dice"); plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# Testing / Evaluation + 3D Side-by-Side Visualization (Plotly)
# =========================================
import torch
import numpy as np
import torch.nn.functional as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm import tqdm
import nibabel as nib

print("🔍 Loading best model for testing...")
model.load_state_dict(torch.load(save_path, map_location=device))
model = model.to(device).float().eval()

test_loss, test_dice = 0.0, 0.0
MAX_VIS = 3     # show only a few samples for clarity

def _plot_3d_pair(pred_vol, tgt_vol, batch_idx, thr_pred=0.3, thr_tgt=0.5):
    """Render prediction and target CT side-by-side."""
    # normalize to [0,1]
    pred = (pred_vol - pred_vol.min()) / (pred_vol.max() - pred_vol.min() + 1e-8)
    tgt  = (tgt_vol - tgt_vol.min()) / (tgt_vol.max() - tgt_vol.min() + 1e-8)
    D, H, W = pred.shape

    def _coords(vol):
        x = np.arange(D).repeat(H*W)
        y = np.tile(np.arange(H).repeat(W), D)
        z = np.tile(np.arange(W), D*H)
        return x, y, z, vol.flatten()

    x_p, y_p, z_p, v_p = _coords(pred)
    x_t, y_t, z_t, v_t = _coords(tgt)

    fig = make_subplots(
        rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]],
        column_widths=[0.5, 0.5], horizontal_spacing=0.05,
        subplot_titles=(f"Prediction (Batch {batch_idx})", "Ground Truth CT")
    )

    # Prediction
    fig.add_trace(go.Isosurface(
        x=x_p, y=y_p, z=z_p, value=v_p,
        isomin=thr_pred, isomax=1.0, surface_count=1,
        colorscale="Viridis", opacity=0.8,
        caps=dict(x_show=False, y_show=False, z_show=False)
    ), row=1, col=1)

    # Ground truth
    fig.add_trace(go.Isosurface(
        x=x_t, y=y_t, z=z_t, value=v_t,
        isomin=thr_tgt, isomax=1.0, surface_count=1,
        colorscale="Gray", opacity=0.7,
        caps=dict(x_show=False, y_show=False, z_show=False)
    ), row=1, col=2)

    fig.update_layout(
        title=f"3D Reconstruction Comparison (Batch {batch_idx})",
        margin=dict(l=0, r=0, t=40, b=0),
        scene=dict(aspectmode="data"),
        scene2=dict(aspectmode="data")
    )
    fig.show()


with torch.no_grad(), torch.amp.autocast("cuda"):
    for b_idx, batch in enumerate(tqdm(small_test_loader, desc="Testing")):
        xa = batch["xa"].to(device, non_blocking=True)
        xb = batch["xb"].to(device, non_blocking=True)
        tgt_raw = batch["vol"].to(device, non_blocking=True)

        # forward
        organ_id = batch.get("organ_id", None)
        output = model(xa, xb, organ_id=organ_id.to(device)) if organ_id is not None else model(xa, xb)

        target = match_target_to_output(tgt_raw, output)
        loss = criterion(output, target)
        dice = compute_dice(output, target, from_logits=True)

        test_loss += loss.item()
        test_dice += dice

        # visualize first few samples
        if b_idx < MAX_VIS:
            probs = torch.sigmoid(output[0,0]).detach().cpu().numpy()
            tgt_np = target[0,0].detach().cpu().numpy()

            # resample target to same shape if necessary
            if tgt_np.shape != probs.shape:
                tgt_t = torch.tensor(tgt_np)[None,None,...]
                tgt_rs = F.interpolate(tgt_t, size=probs.shape, mode="trilinear", align_corners=False)
                tgt_np = tgt_rs[0,0].numpy()

            _plot_3d_pair(probs, tgt_np, b_idx)

avg_test_loss = test_loss / len(small_test_loader)
avg_test_dice = test_dice / len(small_test_loader)
print(f"📊 Test Loss: {avg_test_loss:.4f} | Test Dice: {avg_test_dice:.4f}")


In [ ]:
import torch
from torch import amp
from torchvision import transforms
from PIL import Image

# ======== Config ========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = "/content/drive/MyDrive/saved_models_swinX2S/best_model_01.pth"
img_size = (128, 128)   # <-- set to the size you used in training (e.g., 256, 384, ...)

# If not already defined elsewhere:
# from your_model_lib import SwinX2S
# model = SwinX2S(...)

# ======== Load model weights ========
model = model.to(device).float().eval()
model.load_state_dict(torch.load(ckpt_path, map_location=device))

# ======== Preprocessing (single-channel, resize, tensor) ========
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize(img_size, antialias=True),
    transforms.ToTensor(),                  # -> [1,H,W] in [0,1]
    # If you used normalization during training, add it here:
    # transforms.Normalize(mean=[0.5], std=[0.5]),
])

# ======== Load the AP & LAT images ========
ap_path  = "/content/drive/MyDrive/total_segmentator_refine/femur/p_0001/x-ray/femur_right-ap.png"
lat_path = "/content/drive/MyDrive/total_segmentator_refine/femur/p_0001/x-ray/femur_right-lat.png"

ap_img  = Image.open(ap_path).convert("L")
lat_img = Image.open(lat_path).convert("L")

ap_tensor  = transform(ap_img).unsqueeze(0).to(device)   # [1,1,H,W]
lat_tensor = transform(lat_img).unsqueeze(0).to(device)  # [1,1,H,W]

# ======== Inference (AMP, no grad) ========
with torch.no_grad(), amp.autocast("cuda"):
    # If your SwinX2S forward supports organ_id, pass organ_id=... else just (xa, xb)
    logits = model(ap_tensor, lat_tensor)       # expected [1,1,D,H,W]
    probs  = torch.sigmoid(logits)              # [1,1,D,H,W]

# To numpy for further use/visualization
output_vol = probs.squeeze().detach().cpu().numpy()       # [D,H,W]
print("Output volume shape:", output_vol.shape)

# Optional: binary mask at 0.5 threshold
# import numpy as np
# mask_vol = (output_vol > 0.5).astype(np.uint8)


In [ ]:
import numpy as np
import plotly.graph_objects as go

def render_swinx2s_plotly(volume, threshold=0.5, colorscale="Gray"):
    """
    Render SwinX2S 3D reconstructed output (numpy array [D,H,W]) using Plotly isosurface.

    Args:
        volume: 3D numpy array [D,H,W], typically output from torch.sigmoid(model(...)).cpu().numpy()
        threshold: isosurface threshold (default 0.5)
        colorscale: Plotly color scheme (default 'Gray')
    """
    # Normalize volume to [0,1]
    vol = (volume - np.min(volume)) / (np.max(volume) - np.min(volume) + 1e-8)

    D, H, W = vol.shape
    print(f"Rendering SwinX2S volume with shape: {vol.shape}")

    # Prepare coordinate grid
    x = np.arange(D).repeat(H * W)
    y = np.tile(np.arange(H).repeat(W), D)
    z = np.tile(np.arange(W), D * H)

    # Flatten the volume
    vol_flat = vol.flatten()

    # Create 3D isosurface plot
    fig = go.Figure(data=go.Isosurface(
        x=x,
        y=y,
        z=z,
        value=vol_flat,
        isomin=threshold,
        isomax=1.0,
        surface_count=1,
        colorscale=colorscale,
        caps=dict(x_show=False, y_show=False, z_show=False),
        opacity=0.7
    ))

    # Update figure layout
    fig.update_layout(
        title="SwinX2S 3D Reconstruction",
        scene=dict(
            xaxis_title="Depth (D)",
            yaxis_title="Height (H)",
            zaxis_title="Width (W)",
            aspectmode="data"
        ),
        margin=dict(l=0, r=0, t=40, b=0)
    )

    fig.show()

# Example usage with your model output
render_swinx2s_plotly(output_vol, threshold=0.3, colorscale="Gray")


In [ ]:
import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F
import plotly.graph_objects as go

def render_ct_plotly_resampled(nii_path, target_shape=(64, 128, 128), threshold=0.5):
    """
    Load CT NIfTI volume, resample to target shape (e.g., 64x128x128),
    and render as 3D isosurface using Plotly.
    """
    # Load and normalize CT volume
    vol = nib.load(nii_path).get_fdata().astype(np.float32)
    vol = (vol - vol.min()) / (vol.max() - vol.min() + 1e-8)
    print(f"Original CT volume shape: {vol.shape}")

    # Convert to tensor and add batch/channel dims for interpolation
    vol_t = torch.tensor(vol)[None, None, ...]  # [1,1,D,H,W]
    target_shape_t = target_shape

    # Resample to match SwinX2S output shape
    vol_resampled = F.interpolate(vol_t, size=target_shape_t, mode='trilinear', align_corners=False)
    vol_resampled = vol_resampled[0, 0].cpu().numpy()  # [D,H,W]
    print(f"Resampled CT volume shape: {vol_resampled.shape}")

    # Prepare coordinates
    D, H, W = vol_resampled.shape
    x = np.arange(D).repeat(H * W)
    y = np.tile(np.arange(H).repeat(W), D)
    z = np.tile(np.arange(W), D * H)
    vol_flat = vol_resampled.flatten()

    # Create 3D Plotly isosurface
    fig = go.Figure(data=go.Isosurface(
        x=x,
        y=y,
        z=z,
        value=vol_flat,
        isomin=threshold,
        isomax=1.0,
        surface_count=1,
        colorscale="Gray",
        caps=dict(x_show=False, y_show=False, z_show=False),
        opacity=0.7
    ))

    # Layout
    fig.update_layout(
        title=f"CT Volume (Resampled to {target_shape})",
        scene=dict(
            xaxis_title="Depth (D)",
            yaxis_title="Height (H)",
            zaxis_title="Width (W)",
            aspectmode="data"
        ),
        margin=dict(l=0, r=0, t=40, b=0)
    )

    fig.show()


# Example usage
render_ct_plotly_resampled(
    "/content/drive/MyDrive/total_segmentator_refine/femur/p_0001/ct-scan/femur_right_msk.nii.gz",
    target_shape=(64, 128, 128),
    threshold=0.5
)


In [ ]:
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def render_pred_vs_ct_side_by_side(
    pred_volume,                  # numpy array [D,H,W] from SwinX2S (e.g., 64x128x128)
    ct_nii_path,                  # path to real CT NIfTI
    threshold_pred=0.5,           # isosurface threshold for prediction
    threshold_ct=0.5,             # isosurface threshold for CT
    colorscale_pred="Viridis",
    colorscale_ct="Gray"
):
    """
    Show predicted 3D volume and real CT volume side-by-side with Plotly isosurfaces.
    Resamples CT to match pred_volume shape.
    """
    # Ensure numpy
    pred = np.asarray(pred_volume, dtype=np.float32)
    D, H, W = pred.shape
    target_shape = (D, H, W)

    # Normalize prediction to [0,1]
    pred = (pred - pred.min()) / (pred.max() - pred.min() + 1e-8)

    # Load and normalize CT, then resample to target shape
    ct = nib.load(ct_nii_path).get_fdata().astype(np.float32)
    ct = (ct - ct.min()) / (ct.max() - ct.min() + 1e-8)

    # Torch resample to (D,H,W)
    ct_t = torch.from_numpy(ct)[None, None, ...]                 # [1,1,D,H,W] (whatever original is)
    ct_rs = F.interpolate(ct_t, size=target_shape, mode="trilinear", align_corners=False)
    ct_rs = ct_rs[0, 0].cpu().numpy()

    print(f"Pred shape: {pred.shape} | CT (resampled) shape: {ct_rs.shape}")

    # Prepare coordinates for Plotly
    def _coords_for(vol):
        d, h, w = vol.shape
        x = np.arange(d).repeat(h * w)
        y = np.tile(np.arange(h).repeat(w), d)
        z = np.tile(np.arange(w), d * h)
        return x, y, z, vol.flatten()

    x_p, y_p, z_p, v_p = _coords_for(pred)
    x_c, y_c, z_c, v_c = _coords_for(ct_rs)

    # Two side-by-side 3D scenes
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type":"scene"}, {"type":"scene"}]],
        column_widths=[0.5, 0.5],
        horizontal_spacing=0.04,
        subplot_titles=("Prediction (SwinX2S)", "Real CT (resampled)")
    )

    # Left: Prediction
    fig.add_trace(go.Isosurface(
        x=x_p, y=y_p, z=z_p, value=v_p,
        isomin=threshold_pred, isomax=1.0, surface_count=1,
        colorscale=colorscale_pred,
        caps=dict(x_show=False, y_show=False, z_show=False),
        opacity=0.8,
        name="Prediction"
    ), row=1, col=1)

    # Right: CT
    fig.add_trace(go.Isosurface(
        x=x_c, y=y_c, z=z_c, value=v_c,
        isomin=threshold_ct, isomax=1.0, surface_count=1,
        colorscale=colorscale_ct,
        caps=dict(x_show=False, y_show=False, z_show=False),
        opacity=0.7,
        name="CT"
    ), row=1, col=2)

    # Layout / axes
    fig.update_layout(
        title=f"SwinX2S Prediction vs Real CT (both {target_shape})",
        margin=dict(l=0, r=0, t=50, b=0),
        scene=dict(
            xaxis_title="Depth (D)", yaxis_title="Height (H)", zaxis_title="Width (W)",
            aspectmode="data"
        ),
        scene2=dict(
            xaxis_title="Depth (D)", yaxis_title="Height (H)", zaxis_title="Width (W)",
            aspectmode="data"
        ),
        showlegend=False
    )

    fig.show()


# Example usage:
# Assuming output_vol is your prediction as a numpy array of shape (64,128,128)
render_pred_vs_ct_side_by_side(
    output_vol,
    "/content/drive/MyDrive/total_segmentator_refine/femur/p_0001/ct-scan/femur_right_msk.nii.gz",
    threshold_pred=0.3,
    threshold_ct=0.5
)
